# DSC 2026 Task 2 — LegalQA · A100 Production Pipeline

**Canonical full-run environment.** Kaggle T4×2 is the smoke gate; this notebook is the production run.

| | Kaggle gate | **This notebook (A100)** |
|---|---|---|
| dtype | fp16 (Turing has no bf16) | **bf16 native** |
| gen batch | 4 | **16 (40 GB) / 32 (80 GB)** |
| LLM load | 4-bit NF4 | **merged bf16** (no dequant, no LoRA branch) |
| scope | 20 questions | **1,000 questions** |
| runtime | ~10 min | **~25–40 min** |

**Stack (3.789 B < 4.0 B ceiling):** `Qwen2.5-3B-Instruct` 3.086 B + `huydang-dek21-embedding-v2` 0.135 B + `bge-reranker-v2-m3` 0.568 B

**Run order:** 0.1 → 0.2 → 0.3 → 0.4 → 1.1 → 1.2 → 2.1 → 2.2 → 2.3 → (2.4) → 2.5 → 2.6 → 3.0 → 3.1 → **3.2 gate** → 4.0 → 4.1 → 4.2 → 4.3

Everything heavy caches to Drive. A disconnect costs one stage, never the run.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 0.1 — Environment (A100 / Ampere)
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"]       = "expandable_segments:True"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"]     = "true"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["TOKENIZERS_PARALLELISM"]        = "false"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"]     = "1"   # parallel-stream downloads

!pip install -q "transformers==4.49.0" "peft==0.14.0" "trl==0.15.2" "accelerate==1.4.0"
!pip install -q "bitsandbytes==0.45.3" "sentence-transformers==3.4.1"
!pip install -q "faiss-cpu==1.10.0" "bm25s>=0.2.7,<0.3" "pyvi==0.1.1" "nltk==3.9.1" hf_transfer

import torch, transformers, peft
print("torch:", torch.__version__, "| transformers:", transformers.__version__)
assert torch.cuda.is_available(), "No GPU — Runtime → Change runtime type → A100."

_cap  = torch.cuda.get_device_capability(0)
_name = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
BF16_OK = torch.cuda.is_bf16_supported()

# Ampere (SM 8.0+) → bf16 natively: FP32's exponent range, so no loss scaling
# and no overflow risk on long-context attention. This is the single biggest
# upgrade over the T4 gate, which had to fall back to fp16 + GradScaler.
DTYPE = torch.bfloat16 if BF16_OK else torch.float16
print(f"GPU: {_name} | {VRAM_GB:.0f} GB | SM {_cap[0]}.{_cap[1]} | bf16={BF16_OK} → {DTYPE}")
if not BF16_OK:
    print("⚠ Not an Ampere+ GPU. This notebook is tuned for A100; falling back to fp16 "
          "and you should reduce PROFILE['gen_batch'] in cell 0.2.")

# FlashAttention-2 is ~10-20% faster than SDPA at these shapes but needs a long
# source build on Colab unless a matching wheel exists. Use it only if already
# importable; SDPA's fused kernels are the sane default and cost nothing.
try:
    import flash_attn  # noqa: F401
    ATTN_IMPL = "flash_attention_2"
except ImportError:
    ATTN_IMPL = "sdpa"
print("attention implementation:", ATTN_IMPL)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 0.2 — Drive mount + CFG + A100 profile
# ══════════════════════════════════════════════════════════════════════
import gc, json, re, time, random, shutil, unicodedata, dataclasses
import numpy as np, pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

@dataclasses.dataclass
class CFG:
    data_dir: str = "/content/drive/MyDrive/dsc2026/artifacts-task2"
    work_dir: str = "/content/drive/MyDrive/dsc2026/work"
    scratch_dir: str = "/content/scratch"          # local SSD — fast, ephemeral

    # retriever (swapped for the 4B cap): PhoBERT-base lineage, verified config
    emb_model_id: str = "CODE4LIFEOFFICIAL/huydang-dek21-embedding-v2"
    emb_dim: int = 768                              # config.json hidden_size
    emb_max_len: int = 256                          # 258 positions − 2 RoBERTa offset
    emb_char_budget: int = 900                      # ≈250 tok after pyvi → head-weighted
    emb_truncate_dim: int = 0                       # 0 = full 768 (Matryoshka: 512/256)

    reranker_id: str = "BAAI/bge-reranker-v2-m3"
    llm_id: str = "Qwen/Qwen2.5-3B-Instruct"

    top_k_context: int = 8
    max_parts_per_article: int = 2
    cand_pool: int = 100
    rrf_k: int = 60

    max_chunk_chars: int = 1200                     # 8 × 1200 fits the prompt budget
    max_seq_len: int = 5632                         # MUST scale with K (see 3.1)
    max_new_tokens: int = 1400
    repetition_penalty: float = 1.0                 # 1.0 — see cell 4.0
    sniper_chars: int = 1200

    n_ret_val: int = 400
    n_gen_val: int = 300
    seed: int = 42
    use_known_qa_lookup: bool = True

cfg = CFG()
for d in (cfg.work_dir, cfg.scratch_dir):
    os.makedirs(d, exist_ok=True)

P = lambda *p: os.path.join(cfg.data_dir, *p)       # read-only inputs
W = lambda *p: os.path.join(cfg.work_dir, *p)       # Drive, persisted
S = lambda *p: os.path.join(cfg.scratch_dir, *p)    # local, fast, ephemeral

print(f"data_dir: {cfg.data_dir}\nwork_dir: {cfg.work_dir}")
_missing = [f for f in ("chunks/legal_chunks.parquet", "data/qa_unique.parquet",
                        "data/known_qa.json", "labels/retrieval_labels.parquet",
                        "raw/public-official.json") if not os.path.exists(P(f))]
for f in ("chunks/legal_chunks.parquet", "data/qa_unique.parquet", "data/known_qa.json",
          "labels/retrieval_labels.parquet", "raw/public-official.json"):
    print(f"  {'OK     ' if os.path.exists(P(f)) else 'MISSING'}  {f}")
assert not _missing, f"missing inputs: {_missing}"

# ---- A100 profile ------------------------------------------------------
# KV cache math (Qwen2.5-3B, GQA with 2 KV heads, 36 layers, head_dim 128):
#   36 × 2 × 128 × 2(K+V) × 2(bf16) = 36 KB per token.
#   At K=8 the prompt runs ~5.6k tokens + 1.4k generated ≈ 7k tokens/sequence.
#   batch 16 → 16 × 7k × 36 KB ≈ 4.1 GB  (+ 6.18 GB weights ≈ 10.3 GB)  → 40 GB ✓
#   batch 32 → 32 × 7k × 36 KB ≈ 8.3 GB  (+ 6.18 GB weights ≈ 14.5 GB)  → 80 GB ✓
IS_80GB = VRAM_GB > 60
PROFILE = {
    "emb_batch":    512,
    "rerank_batch": 128,
    "gen_batch":    32 if IS_80GB else 16,
    "st_train_batch": 64,
}
print(f"profile: {PROFILE}")

def set_seed(s=cfg.seed):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

def free_vram(*names):
    g = globals()
    for n in names:
        if n in g: del g[n]
    gc.collect(); torch.cuda.empty_cache()

def vram_report():
    free_b, total_b = torch.cuda.mem_get_info()
    print(f"  torch alloc {torch.cuda.memory_allocated()/1e9:5.2f} GB | "
          f"free {free_b/1e9:5.2f} / {total_b/1e9:.0f} GB")

# ---- HF token (Colab Secrets, not kaggle_secrets) ----------------------
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")             # 🔑 sidebar → Secrets
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("HF token loaded")
else:
    print("No HF_TOKEN (Colab Secrets or env). Public repos still work.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 0.3 — Parameter-budget audit (hard gate)
# ══════════════════════════════════════════════════════════════════════
from transformers import AutoConfig, AutoModel
from huggingface_hub import HfApi

def count_params(model_id):
    """Meta-device instantiation: exact count, no weight download, no VRAM."""
    c = AutoConfig.from_pretrained(model_id)
    with torch.device("meta"):
        m = AutoModel.from_config(c)
    return sum(p.numel() for p in m.parameters())

_api = HfApi(token=HF_TOKEN)
def resolve_revision(rid):
    try:    return _api.model_info(rid).sha
    except Exception: return "unresolved"

BUDGET, MODEL_REVS, total = 4_000_000_000, {}, 0
for role, mid in (("generator", cfg.llm_id), ("retriever", cfg.emb_model_id),
                  ("reranker", cfg.reranker_id)):
    n = count_params(mid); total += n
    MODEL_REVS[role] = {"repo_id": mid, "revision": resolve_revision(mid), "params": n}
    print(f"  {role:<10} {mid:<48} {n/1e9:>6.3f}B  rev {MODEL_REVS[role]['revision'][:10]}")
print(f"  {'TOTAL':<10} {'':<48} {total/1e9:>6.3f}B / 4.000B  "
      f"{'PASS' if total <= BUDGET else 'FAIL'}")
assert total <= BUDGET, f"PARAMETER BUDGET EXCEEDED: {total/1e9:.3f}B > 4.0B"

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 0.4 — Stale-cache invalidation (bge-m3 artifacts poison the new model)
# ══════════════════════════════════════════════════════════════════════
# These hold 768-vs-1024-dim vectors, or ROW POSITIONS chosen by the OLD
# retriever. Same filenames, same schema, silently different meaning — the
# worst kind of stale cache, because nothing errors.
STALE = ["emb_base", "emb_ft", "test_contexts.parquet", "train_contexts.parquet",
         "gen_public.jsonl", "bge_m3_legal_ft"]
# Retriever-independent and expensive to rebuild — never delete these.
PRESERVE = ["qwen25_3b_legal_lora", "bm25_index", "index_texts.pkl.gz",
            "seg_texts.pkl.gz", "seg_texts_emb.pkl.gz", "splits.json"]

CONFIRM_DELETE = False        # ← flip to True after reading the dry-run below

print("STALE (built with bge-m3):")
for s in STALE:
    p = W(s)
    if os.path.exists(p):
        sz = (sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file())
              if os.path.isdir(p) else os.path.getsize(p))
        print(f"  {'DELETED' if CONFIRM_DELETE else 'would delete'}  {s:<28} {sz/1e6:8.1f} MB")
        if CONFIRM_DELETE:
            shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)
    else:
        print(f"  absent        {s}")

print("\nPRESERVED (reusable, retriever-independent):")
for k in PRESERVE:
    p = W(k)
    print(f"  {'OK     ' if os.path.exists(p) else 'absent '}  {k}")

if not CONFIRM_DELETE:
    print("\n→ Review the list, set CONFIRM_DELETE = True, re-run this cell.")

# ⚠ The preserved LoRA adapter was fine-tuned on bge-m3 contexts at K=5/1800
#   chars. Reusing it saves ~2 h and the prompt format is unchanged, but it IS
#   a distribution shift. Cell 3.2 measures it instead of assuming.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 1.1 — Load corpus + QA
# ══════════════════════════════════════════════════════════════════════
CHUNK_COLS = ["chunk_id", "context_id", "doc_id", "document_number", "document_title",
              "name", "structure", "article_number", "article_title", "dieu", "khoan", "content"]
chunks = pd.read_parquet(P("chunks", "legal_chunks.parquet"), columns=CHUNK_COLS)
chunks["content"] = chunks["content"].fillna("")
chunk_ids = chunks["chunk_id"].to_numpy()
chunk_pos = {c: i for i, c in enumerate(chunk_ids)}

qa      = pd.read_parquet(P("data", "qa_unique.parquet"))
qa_by_id = qa.set_index("id")
labels  = pd.read_parquet(P("labels", "retrieval_labels.parquet"))
with open(P("raw", "public-official.json"), encoding="utf-8") as f:
    test_data = json.load(f)
with open(P("data", "known_qa.json"), encoding="utf-8") as f:
    known_qa = json.load(f)

print(f"chunks {chunks.shape} | qa {qa.shape} | labels {labels.shape} | "
      f"test {len(test_data)} | known_qa {len(known_qa['by_question']):,}")
print("structure:", chunks['structure'].value_counts().to_dict())

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 1.2 — Citations → gold chunks, frozen splits
# ══════════════════════════════════════════════════════════════════════
from collections import defaultdict

def norm_key(s):
    """Applied to BOTH sides of the join: citation strings and chunk metadata
    disagree on case, whitespace, and whether 'Điều' prefixes the number."""
    if s is None or (isinstance(s, float) and np.isnan(s)): return ""
    s = unicodedata.normalize("NFC", str(s)).lower().strip()
    s = re.sub(r"^điều\s+", "", s)
    return re.sub(r"\s+", " ", s)

by_doc_art, by_doc = defaultdict(list), defaultdict(list)
for cid, dn, an in zip(chunks["chunk_id"], chunks["document_number"], chunks["article_number"]):
    dn, an = norm_key(dn), norm_key(an)
    if dn:
        by_doc[dn].append(cid)
        if an: by_doc_art[(dn, an)].append(cid)

def resolve_citations(cits):
    gold, keys = [], set()
    for c in (cits if cits is not None else []):
        c = dict(c); dn, an = norm_key(c.get("document_number")), norm_key(c.get("article"))
        if an and (dn, an) in by_doc_art: gold += by_doc_art[(dn, an)]; keys.add((dn, an))
        elif not an and dn in by_doc:     gold += by_doc[dn];           keys.add((dn, ""))
    return list(dict.fromkeys(gold)), keys

labels["gold_chunk_ids"], labels["gold_keys"] = zip(*labels["citations"].map(resolve_citations))
labels["resolvable"] = labels["gold_chunk_ids"].str.len() > 0
labels_by_qid = labels.set_index("qa_id")
print(f"resolvable: {labels['resolvable'].sum()}/{len(labels)} "
      f"({labels['resolvable'].mean():.1%})  ← expect ≈56%")

chunk_keys = [(norm_key(dn), norm_key(an))
              for dn, an in zip(chunks["document_number"], chunks["article_number"])]

sp = W("splits.json")
if os.path.exists(sp):
    splits = json.load(open(sp)); print("loaded frozen splits from Drive")
else:
    rng = np.random.default_rng(cfg.seed)
    res = labels.loc[labels["resolvable"], "qa_id"].tolist()
    ret_val = list(rng.choice(res, size=cfg.n_ret_val, replace=False))
    rest = [q for q in labels["qa_id"] if q not in set(ret_val)]
    splits = {"ret_val": ret_val,
              "gen_val": list(rng.choice(rest, size=cfg.n_gen_val, replace=False))}
    json.dump(splits, open(sp, "w"))
RET_VAL, GEN_VAL = set(splits["ret_val"]), set(splits["gen_val"])
TRAIN_QIDS = [q for q in labels["qa_id"] if q not in RET_VAL and q not in GEN_VAL]
print(f"train {len(TRAIN_QIDS)} | ret_val {len(RET_VAL)} | gen_val {len(GEN_VAL)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 2.1 — BM25 over pyvi-segmented text (reused from Drive if present)
# ══════════════════════════════════════════════════════════════════════
import pickle, gzip, bm25s
from pyvi import ViTokenizer

def has_val(x):
    """Missing metadata arrives as None, NaN, '' or the literal 'None'."""
    if x is None or (isinstance(x, float) and np.isnan(x)): return False
    return str(x).strip() not in ("", "None", "nan")

def title_of(row):
    if has_val(row["document_title"]): return str(row["document_title"])
    t = re.sub(r"[-_]+", " ", str(row["name"] or "")).strip()
    return re.sub(r"\s*\d{4,}\s*$", "", t)          # drop the trailing site id

def index_text(row):
    parts = [title_of(row)]
    if row["structure"] == "dieu" and has_val(row["dieu"]): parts.append(str(row["dieu"]))
    parts.append(row["content"])
    return " . ".join(p for p in parts if p)

IT, ST = W("index_texts.pkl.gz"), W("seg_texts.pkl.gz")
if os.path.exists(IT) and os.path.exists(ST):
    with gzip.open(IT) as f: index_texts = pickle.load(f)
    with gzip.open(ST) as f: seg_texts = pickle.load(f)
    print(f"loaded cached index/seg texts ({len(index_texts):,})")
else:
    index_texts = [index_text(r) for _, r in tqdm(chunks.iterrows(), total=len(chunks), desc="compose")]
    seg_texts = [ViTokenizer.tokenize(t.lower()) for t in tqdm(index_texts, desc="pyvi (bm25)")]
    with gzip.open(IT, "wb") as f: pickle.dump(index_texts, f)
    with gzip.open(ST, "wb") as f: pickle.dump(seg_texts, f)

BM25_DIR = W("bm25_index")
if os.path.exists(os.path.join(BM25_DIR, "params.index.json")):
    bm25 = bm25s.BM25.load(BM25_DIR, mmap=False); print("loaded BM25 index")
else:
    bm25 = bm25s.BM25(k1=0.9, b=0.4)
    bm25.index(bm25s.tokenize(seg_texts, stopwords=None, show_progress=True))
    bm25.save(BM25_DIR)

def bm25_search(queries, k=None):
    k = k or cfg.cand_pool
    seg_q = [ViTokenizer.tokenize(q.lower()) for q in queries]
    idx, sc = bm25.retrieve(bm25s.tokenize(seg_q, stopwords=None, show_progress=False),
                            k=k, show_progress=False)
    return np.asarray(idx), np.asarray(sc)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 2.2 — Case-preserving segmentation for the new encoder
# ══════════════════════════════════════════════════════════════════════
# BM25 wants lowercase. A PhoBERT-family encoder does NOT: its BPE is
# case-sensitive and legal Vietnamese carries signal in capitalisation
# ("Nghị định", "Điều", proper nouns). Two different views, cached separately.
SEG_EMB = W("seg_texts_emb.pkl.gz")
if os.path.exists(SEG_EMB):
    with gzip.open(SEG_EMB) as f: seg_texts_emb = pickle.load(f)
    print(f"loaded encoder segmentation ({len(seg_texts_emb):,})")
else:
    seg_texts_emb = [ViTokenizer.tokenize(t) for t in tqdm(index_texts, desc="pyvi (encoder)")]
    with gzip.open(SEG_EMB, "wb") as f: pickle.dump(seg_texts_emb, f)

def seg_query(q): return ViTokenizer.tokenize(q)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 2.3 — Dense index with huydang-dek21 (768-dim, 256-token ceiling)
# ══════════════════════════════════════════════════════════════════════
import faiss
from sentence_transformers import SentenceTransformer

# HEAD-WEIGHTED ENCODING. The encoder reads 256 tokens; a 2,500-char article is
# ~690 tokens, so a naive encode discards ~63% of it silently. index_texts is
# ordered "<title> . <Điều heading> . <body>", so the surviving window holds the
# document number and article heading — exactly what legal queries match on.
# We cap characters explicitly so the truncation point is predictable.
def encode_corpus(model_id, tag, texts, truncate_dim=0):
    shard_dir = W(f"emb_{tag}"); os.makedirs(shard_dir, exist_ok=True)
    SHARD, n = 32768, len(texts)                     # few large shards: Drive is
    n_shards = (n + SHARD - 1) // SHARD              # slow with many small files
    todo = [s for s in range(n_shards) if not os.path.exists(f"{shard_dir}/s{s:04d}.npy")]
    if todo:
        m = SentenceTransformer(model_id, model_kwargs={"torch_dtype": DTYPE},
                                truncate_dim=(truncate_dim or None), device="cuda")
        m.max_seq_length = cfg.emb_max_len           # 256 — positions stop at 258
        print(f"  max_seq_length={m.max_seq_length} dim={m.get_sentence_embedding_dimension()}")
        for s in tqdm(todo, desc=f"encode[{tag}]"):
            batch = [t[:cfg.emb_char_budget] for t in texts[s*SHARD:(s+1)*SHARD]]
            e = m.encode(batch, batch_size=PROFILE["emb_batch"], normalize_embeddings=True,
                         convert_to_numpy=True, show_progress_bar=False)
            np.save(f"{shard_dir}/s{s:04d}.npy", e.astype(np.float16))
        del m; gc.collect(); torch.cuda.empty_cache()
    embs = np.concatenate([np.load(f"{shard_dir}/s{s:04d}.npy") for s in range(n_shards)])
    assert len(embs) == n, f"shards total {len(embs)} != corpus {n}"
    return embs

def build_faiss(embs):
    """Dimension comes FROM THE DATA — never hardcoded — so a model swap or a
    Matryoshka truncation can't cause a silent dimension mismatch."""
    dim = embs.shape[1]
    assert dim == (cfg.emb_truncate_dim or cfg.emb_dim), \
        f"embedding dim {dim} != configured {cfg.emb_truncate_dim or cfg.emb_dim}"
    ix = faiss.IndexFlatIP(dim)
    ix.add(np.ascontiguousarray(embs, dtype=np.float32))
    print(f"FAISS IndexFlatIP dim={dim} ntotal={ix.ntotal:,} "
          f"({ix.ntotal*dim*4/1e9:.2f} GB RAM)")
    return ix

corpus_emb  = encode_corpus(cfg.emb_model_id, "dek21", seg_texts_emb, cfg.emb_truncate_dim)
faiss_index = build_faiss(corpus_emb)

query_encoder = SentenceTransformer(cfg.emb_model_id, model_kwargs={"torch_dtype": DTYPE},
                                    truncate_dim=(cfg.emb_truncate_dim or None), device="cuda")
query_encoder.max_seq_length = 64

def dense_search(queries, k=None, index=None):
    # NOTE THE SEGMENTATION — skipping it silently costs ~10 recall points.
    k = k or cfg.cand_pool
    ix = index if index is not None else faiss_index
    q = query_encoder.encode([seg_query(x) for x in queries], batch_size=256,
                             normalize_embeddings=True, convert_to_numpy=True,
                             show_progress_bar=False)
    sc, idx = ix.search(np.ascontiguousarray(q, dtype=np.float32), k)
    return idx, sc

_i, _s = dense_search(["Vận chuyển động vật ra khỏi địa bàn cấp tỉnh không có giấy kiểm dịch bị phạt thế nào?"], k=3)
print("dense smoke test:")
for r, (i, s) in enumerate(zip(_i[0], _s[0]), 1):
    print(f"  {r}. {s:.3f}  {chunk_ids[i]}  {index_texts[i][:85]}…")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 2.4 — OPTIONAL: fine-tune the retriever (~12 min on A100)
# ══════════════════════════════════════════════════════════════════════
# A 135M Vietnamese-only encoder capped at 256 tokens needs in-domain signal to
# match the 568M multilingual model it replaced. At 135M this is ~4× cheaper
# than the old bge-m3 fine-tune, so it is worth doing rather than skipping.
RUN_RETRIEVER_FT = True

from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss, MatryoshkaLoss
from datasets import Dataset as HFDataset

FT_DIR = W("dek21_legal_ft")
if RUN_RETRIEVER_FT and not os.path.exists(os.path.join(FT_DIR, "config.json")):
    tr = labels_by_qid.loc[[q for q in TRAIN_QIDS if labels_by_qid.at[q, "resolvable"]]]
    b_idx, _ = bm25_search(tr["query"].tolist(), k=200)
    A, Pp, N = [], [], []
    for (qid, row), cand in zip(tr.iterrows(), b_idx):
        gold, cited = set(row["gold_chunk_ids"]), {dn for dn, _ in row["gold_keys"]}
        cand = [int(c) for c in cand]
        pp = next((c for c in cand if chunk_ids[c] in gold), None)
        pid = chunk_ids[pp] if pp is not None else row["gold_chunk_ids"][0]
        # Hard negatives exclude ALL chunks of cited documents: other parts of
        # the correct Điều are genuinely relevant, and pushing them away would
        # damage retrieval of exactly the text the generator needs.
        for nneg in [c for c in cand if chunk_keys[c][0] not in cited][:2]:
            A.append(seg_query(row["query"]))
            Pp.append(seg_texts_emb[chunk_pos[pid]][:cfg.emb_char_budget])
            N.append(seg_texts_emb[nneg][:cfg.emb_char_budget])
    triplets = HFDataset.from_dict({"anchor": A, "positive": Pp, "negative": N})
    print(f"mined {len(triplets):,} triplets")

    free_vram("query_encoder")
    st = SentenceTransformer(cfg.emb_model_id, device="cuda")
    st.max_seq_length = cfg.emb_max_len
    loss = MatryoshkaLoss(st, MultipleNegativesRankingLoss(st), matryoshka_dims=[768, 512, 256])
    SentenceTransformerTrainer(
        model=st,
        args=SentenceTransformerTrainingArguments(
            output_dir=S("st_runs"), num_train_epochs=3,
            per_device_train_batch_size=PROFILE["st_train_batch"],
            learning_rate=2e-5, warmup_ratio=0.1,
            bf16=BF16_OK, fp16=not BF16_OK,
            logging_steps=50, save_strategy="no", report_to="none"),
        train_dataset=triplets, loss=loss).train()
    st.save_pretrained(FT_DIR)
    del st, loss; gc.collect(); torch.cuda.empty_cache()

if RUN_RETRIEVER_FT and os.path.exists(os.path.join(FT_DIR, "config.json")):
    corpus_emb = encode_corpus(FT_DIR, "dek21_ft", seg_texts_emb, cfg.emb_truncate_dim)
    faiss_index = build_faiss(corpus_emb)
    free_vram("query_encoder")
    query_encoder = SentenceTransformer(FT_DIR, model_kwargs={"torch_dtype": DTYPE},
                                        truncate_dim=(cfg.emb_truncate_dim or None), device="cuda")
    query_encoder.max_seq_length = 64
    print("using the fine-tuned retriever")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 2.5 — RRF fusion + rerank + diversify + reorder  (retrieve_k)
# ══════════════════════════════════════════════════════════════════════
from sentence_transformers import CrossEncoder

def rrf_fuse(rank_lists, k_out, k_rrf=None):
    """score(d) = Σ 1/(60 + rank). No score calibration needed — BM25 scores and
    cosine similarities live on incomparable scales."""
    k_rrf = k_rrf or cfg.rrf_k
    sc = defaultdict(float)
    for ranks in rank_lists:
        for r, pos in enumerate(ranks): sc[int(pos)] += 1.0 / (k_rrf + r + 1)
    return [p for p, _ in sorted(sc.items(), key=lambda x: -x[1])[:k_out]]

def hybrid_search(queries, k=None):
    k = k or cfg.cand_pool
    b, _ = bm25_search(queries, k=cfg.cand_pool)
    d, _ = dense_search(queries, k=cfg.cand_pool)
    return [rrf_fuse([b[i], d[i]], k_out=k) for i in range(len(queries))]

def diversify(positions, max_parts=None):
    """Cap parts-per-(document, article): K slots should buy K distinct
    provisions, not one long Điều repeated across its _p1.._pN pieces."""
    max_parts = max_parts or cfg.max_parts_per_article
    seen, out = defaultdict(int), []
    for p in positions:
        key = chunk_keys[int(p)]
        if seen[key] < max_parts: seen[key] += 1; out.append(int(p))
    return out

def reorder_lost_in_middle(positions):
    """Liu et al. 2023: decoders attend most to the START and END of a long
    context. Rank 1 first, rank 2 LAST, rank 3 second… so the two strongest
    chunks occupy the two high-attention slots. Free; never hurts."""
    head, tail = [], []
    for i, p in enumerate(positions): (head if i % 2 == 0 else tail).append(p)
    return head + tail[::-1]

reranker = CrossEncoder(cfg.reranker_id, max_length=512, device="cuda",
                        automodel_args={"torch_dtype": DTYPE})

def retrieve_k(queries, k=None):
    k = k or cfg.top_k_context
    fused, out = hybrid_search(queries, k=cfg.cand_pool), []
    for q, cand in zip(tqdm(queries, desc="rerank", leave=False), fused):
        pairs = [(q, index_texts[p][:2500]) for p in cand]
        sc = reranker.predict(pairs, batch_size=PROFILE["rerank_batch"], show_progress_bar=False)
        ranked = [p for p, _ in sorted(zip(cand, sc), key=lambda x: -x[1])]
        out.append(reorder_lost_in_middle(diversify(ranked)[:k]))
    return out

# ---- article-level retrieval eval (is the new stack actually good?) ------
def hit(pos, gold_keys):
    dn, an = chunk_keys[pos]
    return (dn, an) in gold_keys or (dn, "") in gold_keys

def eval_retrieval(fn, name, ks=(1, 3, 5, 8)):
    val = labels_by_qid.loc[list(RET_VAL)[:200]]
    ranked = fn(val["query"].tolist(), max(ks))
    hits, mrr = {k: 0 for k in ks}, 0.0
    for pos, gk in zip(ranked, val["gold_keys"]):
        f = next((r for r, p in enumerate(pos) if hit(p, gk)), None)
        if f is not None:
            mrr += 1.0 / (f + 1)
            for k in ks:
                if f < k: hits[k] += 1
    row = {f"hit@{k}": hits[k]/len(val) for k in ks}; row["MRR"] = mrr/len(val)
    print(f"{name:>22}: " + "  ".join(f"{m}={v:.3f}" for m, v in row.items()))
    return row

m_hybrid = eval_retrieval(lambda q, k: [list(r) for r in hybrid_search(q, k)], "hybrid RRF")
m_rerank = eval_retrieval(retrieve_k, "hybrid+rerank (K=8)")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 2.6 — Build test contexts (1,000 queries) then free the retrieval stack
# ══════════════════════════════════════════════════════════════════════
CTX = W(f"test_contexts_k{cfg.top_k_context}.parquet")
if os.path.exists(CTX):
    test_ctx = pd.read_parquet(CTX).set_index("qa_id"); print(f"loaded {test_ctx.shape}")
else:
    qids = list(test_data); questions = [test_data[q]["question"] for q in qids]
    pos, B = [], 128
    for i in tqdm(range(0, len(questions), B), desc=f"retrieve K={cfg.top_k_context}"):
        pos += retrieve_k(questions[i:i+B])
    test_ctx = pd.DataFrame({"qa_id": qids, "question": questions,
                             "ctx_positions": [list(map(int, p)) for p in pos]})
    test_ctx.to_parquet(CTX); test_ctx = test_ctx.set_index("qa_id")

# Contexts for the generation-val split too — cell 3.2 needs them.
VCTX = W(f"val_contexts_k{cfg.top_k_context}.parquet")
if os.path.exists(VCTX):
    val_ctx = pd.read_parquet(VCTX).set_index("qa_id")
else:
    vq = list(GEN_VAL)[:120]
    vquestions = [qa_by_id.at[q, "question"] for q in vq]
    vpos = []
    for i in tqdm(range(0, len(vq), 128), desc="retrieve val"):
        vpos += retrieve_k(vquestions[i:i+128])
    val_ctx = pd.DataFrame({"qa_id": vq, "question": vquestions,
                            "ctx_positions": [list(map(int, p)) for p in vpos]})
    val_ctx.to_parquet(VCTX); val_ctx = val_ctx.set_index("qa_id")
print(f"test_ctx {test_ctx.shape} | val_ctx {val_ctx.shape}")

# Retrieval stack is done — evict ~2.5 GB before the LLM loads.
free_vram("reranker", "query_encoder", "corpus_emb", "faiss_index")
vram_report()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 3.0 — Load Qwen2.5-3B + LoRA, MERGED to bf16
# ══════════════════════════════════════════════════════════════════════
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

ADAPTER_DIR = W("qwen25_3b_legal_lora")
assert os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")), \
    f"LoRA adapter not found at {ADAPTER_DIR}"

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_DIR if os.path.exists(os.path.join(ADAPTER_DIR, "tokenizer_config.json")) else cfg.llm_id)
tokenizer.padding_side    = "left"     # batched decoder-only generation
tokenizer.truncation_side = "left"     # sacrifice leading context, never the question

# On an A100, 4-bit is the WRONG trade: NF4 dequantizes every weight tile on
# every forward pass to buy memory we already have. Merged bf16 removes both the
# dequant cost AND the 252 extra LoRA matmuls per forward (7 projections × 36
# layers), since merge_and_unload folds BA into W. 6.18 GB of weights.
model = AutoModelForCausalLM.from_pretrained(
    cfg.llm_id, torch_dtype=DTYPE, attn_implementation=ATTN_IMPL,
    device_map={"": 0},          # pin to cuda:0. "auto" silently offloads layers
)                                # to CPU under pressure → ~100× slower generation
model = PeftModel.from_pretrained(model, ADAPTER_DIR, torch_dtype=DTYPE)
model = model.merge_and_unload()
model.eval(); model.config.use_cache = True

_devs = {p.device.type for p in model.parameters()}
assert _devs == {"cuda"}, f"CPU offloading detected: {_devs} — refusing to run slow."
print(f"model on GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB | attn={ATTN_IMPL}")
vram_report()

# NOTE: the adapter was trained against NF4-quantised base weights, so
# W_nf4 + BA ≠ W_bf16 + BA exactly. This is the standard QLoRA deployment path
# and quality holds in practice — cell 3.2 verifies rather than assumes.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 3.1 — Sniper clause extractor + SYS_B + prompt renderer
# ══════════════════════════════════════════════════════════════════════
def chunk_header(pos):
    row = chunks.iloc[pos]; t = title_of(row)
    if row["structure"] == "dieu" and has_val(row["dieu"]):
        art = str(row["dieu"]).split(".")[0].strip()
        kh = f", khoản {row['khoan']}" if has_val(row["khoan"]) else ""
        return f"{art}{kh} — {t}"
    return t

_MARK = re.compile(r"(?:^|(?<=\n)|(?<=[.;:])\s|(?<=\s))\s*((?:\d{1,2}\.)|(?:[a-zA-ZđĐ]\)))(?=\s+\S)")

def split_clauses(text):
    """-> [(marker|None, segment)] in DOCUMENT ORDER; element 0 is the preamble.
    Markers are accepted only in ASCENDING runs (1. 2. 3. / a) b) c)), so a
    money amount like '5.000.000 đồng' cannot be mistaken for Khoản 5."""
    marks = [(m.start(1), m.group(1)) for m in _MARK.finditer(text)]
    keep, last_n, last_a = [], 0, ""
    for s, g in marks:
        if g[0].isdigit():
            n = int(g[:-1])
            if n == last_n + 1 or (n == 1 and last_n == 0):
                keep.append((s, g)); last_n = n; last_a = ""
        else:
            ch = g[0].lower()
            if last_a == "" or ord(ch) > ord(last_a): keep.append((s, g)); last_a = ch
    if not keep: return [(None, text)]
    out = []
    if keep[0][0] > 0: out.append((None, text[:keep[0][0]].strip()))
    for i, (s, g) in enumerate(keep):
        end = keep[i+1][0] if i+1 < len(keep) else len(text)
        out.append((g, text[s:end].strip()))
    return [(m, t) for m, t in out if t]

VI_STOP = set("""và của có là được các những cho với trong khi thì mà này đó nếu về như theo
tại từ đến ra vào một hai người việc phải sẽ đã không hay hoặc bị do nào gì thế bao nhiêu ai
sao quy định trường hợp thực hiện đối tượng bạn tôi hỏi xin cảm ơn ạ vậy còn cũng nên""".split())

def q_terms(q):
    t = re.sub(r"[^\w\s/\-]", " ", unicodedata.normalize("NFC", q.lower()))
    return [w for w in t.split() if w not in VI_STOP and len(w) > 1]

def _score_segment(seg, terms):
    s = unicodedata.normalize("NFC", seg.lower())
    hits = sum(1 for t in terms if t in s)
    cov = hits / max(1, len(set(terms)))
    return cov * (1.0 + 0.15*np.log1p(hits)) / (1.0 + len(seg)/4000.0)

def sniper_extract_clauses(text, question, budget=None):
    """Append ONLY the query-relevant Khoản/Điểm, reassembled IN DOCUMENT ORDER.
    Replaces the blind content[:1500] dump, which raised METEOR recall but cost
    ~0.054 ROUGE-L precision. Order matters: ROUGE-L is an LCS metric, so
    shuffling clauses destroys it even when every token survives."""
    budget = budget or cfg.sniper_chars
    segs = split_clauses(text)
    if len(segs) == 1 and segs[0][0] is None: return text[:budget]
    terms = q_terms(question)
    scored = sorted(((_score_segment(s, terms) + (10.0 if m is None else 0.0), i)
                     for i, (m, s) in enumerate(segs)), reverse=True)
    chosen, used = set(), 0
    for sc, i in scored:
        seg = segs[i][1]
        if used + len(seg) > budget and chosen: continue
        chosen.add(i); used += len(seg)
        if used >= budget: break
    return "\n".join(segs[i][1] for i in sorted(chosen))

SYS_B = (
    "Bạn là trợ lý pháp lý chuyên về pháp luật Việt Nam. Nhiệm vụ của bạn là trả lời "
    "câu hỏi của người dùng DỰA TRÊN các trích đoạn văn bản pháp luật được cung cấp.\n"
    "Yêu cầu bắt buộc về cách trả lời:\n"
    "- Mở đầu bằng việc nêu căn cứ pháp lý (ví dụ: \"Căn cứ khoản 3 Điều 17 Nghị định 90/2017/NĐ-CP...\").\n"
    "- Trích dẫn ĐẦY ĐỦ, nguyên văn TẤT CẢ các khoản, điểm có liên quan đến câu hỏi "
    "(kể cả khi có nhiều khoản), giữ nguyên cách đánh số 1., 2., a), b) như trong văn bản. "
    "Không tóm tắt, không rút gọn nội dung điều luật. Sau khi trích dẫn đầy đủ mới đưa ra kết luận.\n"
    "- Trả lời mạch lạc bằng văn xuôi tiếng Việt, không bịa đặt "
    "điều khoản không có trong ngữ cảnh, không nhắc đến việc bạn được cung cấp ngữ cảnh."
)

def to_infer_text(question, positions):
    blocks = [f"[Văn bản {i}] {chunk_header(int(p))}\n{chunks.iloc[int(p)]['content'][:cfg.max_chunk_chars]}"
              for i, p in enumerate(positions, 1)]
    user = ("Các trích đoạn văn bản pháp luật:\n\n" + "\n\n".join(blocks) +
            f"\n\nCâu hỏi: {question.strip()}\n\nTrả lời:")
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": SYS_B}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True)

# ---- self-tests: fail loudly rather than degrade silently ---------------
_t = ("Điều 17. Vi phạm kiểm dịch. " + " ".join(f"{i}. Nội dung không liên quan." for i in range(1, 6))
      + " 6. Phạt tiền từ 5.000.000 đồng đến 6.000.000 đồng đối với hành vi vận chuyển động vật "
        "ra khỏi địa bàn cấp tỉnh mà không có Giấy chứng nhận kiểm dịch.")
_o = sniper_extract_clauses(_t, "vận chuyển động vật ra khỏi địa bàn cấp tỉnh phạt bao nhiêu", 420)
assert "5.000.000" in _o and "5.000.000" not in _t[:420], "sniper regression"
assert _o.index("Điều 17") == 0, "document order not preserved"

# The prompt MUST fit: 8 blocks × 1200 chars + answer. If this trips, raise
# cfg.max_seq_len — otherwise the tokenizer silently left-truncates chunks away.
_q0 = test_ctx.index[0]
_n = len(tokenizer(to_infer_text(test_ctx.at[_q0, "question"],
                                 test_ctx.at[_q0, "ctx_positions"]), add_special_tokens=False).input_ids)
print(f"sniper OK | sample prompt {_n} tokens (cap {cfg.max_seq_len})")
assert _n <= cfg.max_seq_len, f"prompt {_n} > max_seq_len {cfg.max_seq_len} — raise it"

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 3.2 — ⛔ GO / NO-GO GATE: measure before spending the full run
# ══════════════════════════════════════════════════════════════════════
# Three unvalidated changes are stacked here: a new retriever, K=8 (the adapter
# was trained at K=5/1800), and the Sniper append. Each is individually sound;
# together they could regress. 120 val questions ≈ 4 min on an A100 — cheap
# insurance against burning a submission slot and 35 minutes.
RUN_VAL_GATE = True
BASELINE_METEOR = 0.5487        # the score this pipeline must beat

def vi_tokens(s): return unicodedata.normalize("NFC", str(s).lower()).split()

def _align(h, r):
    he, rr, m = list(enumerate(h)), list(enumerate(r)), []
    for i in range(len(he)-1, -1, -1):
        for j in range(len(rr)-1, -1, -1):
            if he[i][1] == rr[j][1]: m.append((he[i][0], rr[j][0])); he.pop(i); rr.pop(j); break
    return sorted(m)

def _chunks_of(m):
    if not m: return 0
    c = 1
    for i in range(len(m)-1):
        if not (m[i+1][0] == m[i][0]+1 and m[i+1][1] == m[i][1]+1): c += 1
    return c

def meteor(hyp, ref, alpha=0.9, beta=3.0, gamma=0.5):
    """NLTK's METEOR minus the stem/synonym stages — which are English-only
    (Porter + WordNet) and fire on exactly ZERO Vietnamese tokens. Identical
    behaviour here, no nltk data download to fail on."""
    h, r = vi_tokens(hyp), vi_tokens(ref)
    if not h or not r: return 0.0
    m = _align(h, r); mm = len(m)
    if not mm: return 0.0
    Pp, R = mm/len(h), mm/len(r)
    return (Pp*R/(alpha*Pp + (1-alpha)*R)) * (1 - gamma*(_chunks_of(m)/mm)**beta)

def rouge_l(hyp, ref):
    h, r = vi_tokens(hyp), vi_tokens(ref)
    if not h or not r: return 0.0
    prev = [0]*(len(r)+1)
    for i in range(1, len(h)+1):
        cur = [0]*(len(r)+1)
        for j in range(1, len(r)+1):
            cur[j] = prev[j-1]+1 if h[i-1] == r[j-1] else max(prev[j], cur[j-1])
        prev = cur
    p, rc = prev[-1]/len(h), prev[-1]/len(r)
    return 0.0 if p+rc == 0 else 2*p*rc/(p+rc)

@torch.inference_mode()
def generate_answers(prompts, max_new_tokens=None, batch_size=None):
    mnt = max_new_tokens or cfg.max_new_tokens
    bs  = batch_size or PROFILE["gen_batch"]
    out = []
    for i in tqdm(range(0, len(prompts), bs), desc="generate", leave=False):
        enc = tokenizer(prompts[i:i+bs], return_tensors="pt", padding=True,
                        truncation=True, max_length=cfg.max_seq_len).to(model.device)
        gen = model.generate(**enc, do_sample=False, max_new_tokens=mnt,
                             repetition_penalty=cfg.repetition_penalty,
                             pad_token_id=tokenizer.pad_token_id)
        out += tokenizer.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in out]

if RUN_VAL_GATE:
    vids = list(val_ctx.index)
    t0 = time.time()
    hyps = generate_answers([to_infer_text(val_ctx.at[q, "question"],
                                           val_ctx.at[q, "ctx_positions"]) for q in vids])
    finals = [h + "\n\nTrích dẫn quy định:\n" +
              sniper_extract_clauses(chunks.iloc[int(val_ctx.at[q, "ctx_positions"][0])]["content"],
                                     val_ctx.at[q, "question"])
              for q, h in zip(vids, hyps)]
    refs = [qa_by_id.at[q, "answer"] for q in vids]
    M  = float(np.mean([meteor(h, r)  for h, r in zip(finals, refs)]))
    RL = float(np.mean([rouge_l(h, r) for h, r in zip(finals, refs)]))
    sec_q = (time.time()-t0)/len(vids)
    print(f"\n{'='*58}\nVAL (n={len(vids)}):  METEOR {M:.4f}   ROUGE-L {RL:.4f}")
    print(f"mean {np.mean([len(f.split()) for f in finals]):.0f} words | "
          f"{sec_q:.1f} s/question → full 1,000 ≈ {sec_q*1000/60:.0f} min")
    print(f"baseline to beat: {BASELINE_METEOR}\n{'='*58}")
    if M < BASELINE_METEOR - 0.01:
        print("⛔ NO-GO: below baseline. Ablate before the full run —\n"
              "   try cfg.top_k_context=5 + cfg.max_chunk_chars=1800 (the trained config),\n"
              "   then RUN_RETRIEVER_FT, then sniper_chars. Change ONE thing at a time.")
    else:
        print("✅ GO — proceed to cell 4.0.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 4.0 — Full batched inference, 1,000 questions (resume-safe)
# ══════════════════════════════════════════════════════════════════════
GEN_OUT = W("gen_public_a100.jsonl")     # NEW filename: never blend configs

done = {}
if os.path.exists(GEN_OUT):
    with open(GEN_OUT, encoding="utf-8") as f:
        for line in f:
            r = json.loads(line); done[r["qa_id"]] = r["answer"]
    print(f"resuming: {len(done)} already generated")

todo = [q for q in test_ctx.index if q not in done]
prompts = {q: to_infer_text(test_ctx.at[q, "question"], test_ctx.at[q, "ctx_positions"])
           for q in todo}
todo.sort(key=lambda q: -len(prompts[q]))    # length-sorted batches → less pad waste

B, t0 = PROFILE["gen_batch"], time.time()
with open(GEN_OUT, "a", encoding="utf-8") as f:
    for i in tqdm(range(0, len(todo), B), desc="public test"):
        qb = todo[i:i+B]
        # greedy: overlap metrics reward fidelity, not diversity.
        # repetition_penalty=1.0 is deliberate — HF applies the penalty over
        # input_ids INCLUDING the prompt, so anything above 1.0 suppresses the
        # retrieved statute, i.e. exactly the tokens the metric pays for.
        for qid, ans in zip(qb, generate_answers([prompts[q] for q in qb])):
            src = sniper_extract_clauses(
                chunks.iloc[int(test_ctx.at[qid, "ctx_positions"][0])]["content"],
                test_ctx.at[qid, "question"])
            final = ans.strip() + "\n\nTrích dẫn quy định:\n" + src
            f.write(json.dumps({"qa_id": qid, "answer": final}, ensure_ascii=False) + "\n")
            done[qid] = final
        f.flush()                             # a disconnect costs one batch
print(f"generated {len(done)}/{len(test_ctx)} in {(time.time()-t0)/60:.1f} min")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 4.1 — Known-QA override + never-empty fallback
# ══════════════════════════════════════════════════════════════════════
# 41 public questions appear VERBATIM in the organiser-provided training data.
# Substituting the organiser's own reference is legitimate use of provided data.
# Exact normalized matching only: a wrong fuzzy match scores ~0, so a loose
# threshold has negative expected value unless validated on held-out data.
norm_q = lambda s: re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s)).lower().strip())
known_by_q = {norm_q(k): v for k, v in known_qa["by_question"].items()} if cfg.use_known_qa_lookup else {}

submission, n_known, n_fallback = {}, 0, 0
for qid in test_ctx.index:
    ans = (done.get(qid) or "").strip()
    hitq = known_by_q.get(norm_q(test_ctx.at[qid, "question"]))
    if hitq: ans, n_known = hitq.strip(), n_known + 1
    if not ans:                                  # never submit empty: METEOR = 0
        p0 = int(test_ctx.at[qid, "ctx_positions"][0])
        ans = (f"Căn cứ {chunk_header(p0)}: "
               + sniper_extract_clauses(chunks.iloc[p0]["content"],
                                        test_ctx.at[qid, "question"], 1200))
        n_fallback += 1
    submission[qid] = {"answer": ans}
print(f"known-QA hits: {n_known} (expect 41) | fallbacks: {n_fallback}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 4.2 — Codabench package + integrity assertions
# ══════════════════════════════════════════════════════════════════════
import zipfile, hashlib
assert set(submission) == set(test_data), "QID set mismatch vs public-official.json"
assert len(submission) == 1000, f"expected 1000 answers, got {len(submission)}"
assert all(isinstance(v, dict) and v.get("answer") for v in submission.values()), "empty answer"

SUB_JSON, SUB_ZIP = W("submission.json"), W("submission.json.zip")
with open(SUB_JSON, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=1)
with zipfile.ZipFile(SUB_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(SUB_JSON, arcname="submission.json")
assert json.load(open(SUB_JSON, encoding="utf-8")) == submission, "JSON round-trip mismatch"

_h = hashlib.sha256()
with open(SUB_JSON, "rb") as f:
    while b := f.read(1 << 20): _h.update(b)
SUB_SHA = _h.hexdigest()
print(f"{len(submission)} answers | mean "
      f"{np.mean([len(v['answer'].split()) for v in submission.values()]):.0f} words")
print(f"sha256 {SUB_SHA[:32]}…\n→ upload to Codabench: {SUB_ZIP}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 4.3 — Run bundle (audit parity with the Kaggle gate)
# ══════════════════════════════════════════════════════════════════════
import subprocess, socket, platform, uuid
from datetime import datetime, timezone

BUNDLE = Path(W("run_bundle")); BUNDLE.mkdir(parents=True, exist_ok=True)
RUN_ID = f"{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}-{uuid.uuid4().hex[:8]}"
_sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()

manifest = {
    "run_id": RUN_ID, "environment": "colab-a100", "mode": "FULL",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "gpu": _name, "vram_gb": round(VRAM_GB, 1), "dtype": str(DTYPE), "attn": ATTN_IMPL,
    "models": MODEL_REVS,
    "param_total_b": round(total/1e9, 4),
    "retriever_finetuned": bool(RUN_RETRIEVER_FT and os.path.exists(os.path.join(FT_DIR, "config.json"))),
    "retrieval_eval": {"hybrid": m_hybrid, "hybrid_rerank_k8": m_rerank},
    "val_gate": ({"meteor": M, "rouge_l": RL, "n": len(val_ctx)} if RUN_VAL_GATE else None),
    "config": {k: getattr(cfg, k) for k in
               ("emb_model_id", "emb_max_len", "emb_char_budget", "top_k_context",
                "max_parts_per_article", "cand_pool", "rrf_k", "max_chunk_chars",
                "max_seq_len", "max_new_tokens", "repetition_penalty", "sniper_chars", "seed")},
    "gen_batch": PROFILE["gen_batch"],
    "known_qa_hits": n_known, "fallbacks": n_fallback,
    "submission_sha256": SUB_SHA, "outputs": {"json": SUB_JSON, "zip": SUB_ZIP},
}
(BUNDLE / "run_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
(BUNDLE / "nvidia-smi.txt").write_text(_sh("nvidia-smi"), encoding="utf-8")
(BUNDLE / "environment.txt").write_text(
    f"run_id={RUN_ID}\nhost={socket.gethostname()}\npython={platform.python_version()}\n"
    f"torch={torch.__version__} cuda={torch.version.cuda}\ngpu={_name}\ndtype={DTYPE}\n\n"
    f"=== pip freeze ===\n{_sh('pip freeze')}", encoding="utf-8")

print(f"RUN BUNDLE → {BUNDLE}")
for p in sorted(BUNDLE.iterdir()): print(f"  {p.name:<22} {p.stat().st_size/1024:7.1f} KB")
print(f"\nRUN_ID {RUN_ID}  |  submission sha256 {SUB_SHA[:24]}…")
print("Commit run_bundle/ alongside the leaderboard score.")